<a href="https://colab.research.google.com/github/springboardmentor891v/CreditPathAI/blob/Rajath/notebooks/pre_processing__methods_2_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Preprocessing & Modeling ---
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- Models ---
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from xgboost import XGBClassifier

# --- Evaluation ---
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [23]:

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/My Drive/Loan_Default.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
# Drop columns based on prior domain knowledge
cols_to_drop = [
    'ID', 'year', 'construction_type', 'Secured_by', 'Security_Type',
    'open_credit', 'Upfront_charges', 'Interest_rate_spread',
    'co-applicant_credit_type'
]
df = df.drop(columns=cols_to_drop)

In [25]:
print(f"Original unique values in 'Status' column: {df['Status'].unique()}")

Original unique values in 'Status' column: [1 0]


In [26]:
df['Status'] = pd.to_numeric(df['Status'], errors='coerce')

In [27]:
valid_statuses = [0, 1]
df = df[df['Status'].isin(valid_statuses)]
print(f"Shape after keeping only rows with Status of 0 or 1: {df.shape}")

Shape after keeping only rows with Status of 0 or 1: (148670, 25)


In [28]:
df['Status'] = df['Status'].astype(int)

In [29]:
#daatset splitting
X = df.drop('Status', axis=1)
y = df['Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("\nData successfully split.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print("-" * 50)



Data successfully split.
X_train shape: (111502, 24)
X_test shape: (37168, 24)
--------------------------------------------------


In [30]:
numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

numerical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough'
)

print("\nPreprocessing pipeline built successfully.")
print("-" * 50)



Preprocessing pipeline built successfully.
--------------------------------------------------


In [31]:
scale_pos_weight_value = y_train.value_counts()[0] / y_train.value_counts()[1]

models = {
    "Logistic Regression": LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight='balanced'),
    "XGBoost": XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight_value),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=10),
    "Gaussian Naive Bayes": GaussianNB(),
    "Bernoulli Naive Bayes": BernoulliNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

for name, model in models.items():
    print(f"\n--- Training and Evaluating: {name} ---")

    model_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    model_pipeline.fit(X_train, y_train)
    y_pred = model_pipeline.predict(X_test)

    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f} (Note: Can be misleading!)")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report (for test data):")
    print(classification_report(y_test, y_pred))
    print("-" * 50)



--- Training and Evaluating: Logistic Regression ---
Accuracy: 0.8329 (Note: Can be misleading!)

Confusion Matrix:
[[24982  3026]
 [ 3185  5975]]

Classification Report (for test data):
              precision    recall  f1-score   support

           0       0.89      0.89      0.89     28008
           1       0.66      0.65      0.66      9160

    accuracy                           0.83     37168
   macro avg       0.78      0.77      0.77     37168
weighted avg       0.83      0.83      0.83     37168

--------------------------------------------------

--- Training and Evaluating: Random Forest ---
Accuracy: 0.9331 (Note: Can be misleading!)

Confusion Matrix:
[[27268   740]
 [ 1746  7414]]

Classification Report (for test data):
              precision    recall  f1-score   support

           0       0.94      0.97      0.96     28008
           1       0.91      0.81      0.86      9160

    accuracy                           0.93     37168
   macro avg       0.92      0.89 